In [14]:
using CSV
using DataFrames
using Flux
using Dates
using Statistics
using StatsBase
using Random

# 1. INCLUDIAMO LE UTILS
include("utils.jl") 

# ============================================================================
# 2. CARICAMENTO E PREPARAZIONE DATI
# ============================================================================
println("--- FASE 1: Caricamento e Pulizia Dati ---")

filename = "Fraudulent_E-Commerce_Transaction_Data_merge.csv"
df_original = CSV.read(filename, DataFrame)

# --- Feature Engineering Avanzata ---

# 1. Logaritmo sull'importo (AIUTA TANTISSIMO)
# I soldi hanno distribuzioni strane, il log le appiattisce rendendole leggibili alla rete
df_original."LogAmount" = log.(df_original."Transaction Amount" .+ 1)

# 2. IP Frequency
ip_counts = countmap(df_original."IP Address")
df_original.IP_Frequency = [ip_counts[ip] for ip in df_original."IP Address"]

# 3. Transaction Hour
df_original."Transaction Hour" = Float32.(df_original."Transaction Hour")

# --- Selezione Colonne (Usiamo LogAmount invece di Amount) ---
input_cols = ["LogAmount", "Transaction Hour", "IP_Frequency", "Account Age Days"]
target_col = "Is Fraudulent"
id_col = "Transaction ID"

println("Feature utilizzate: $input_cols")

# ============================================================================
# 3. BILANCIAMENTO DATASET
# ============================================================================
fraud_rows = df_original[df_original[:, target_col] .== 1, :]
legit_rows = df_original[df_original[:, target_col] .== 0, :]

n_min = min(nrow(fraud_rows), nrow(legit_rows))

# Creiamo dataset bilanciato
df_train = vcat(
    fraud_rows[randperm(nrow(fraud_rows))[1:n_min], :],
    legit_rows[randperm(nrow(legit_rows))[1:n_min], :]
)
df_train = df_train[randperm(nrow(df_train)), :]

println("Dataset Bilanciato: $(nrow(df_train)) righe totali.")

train_inputs = Matrix{Float32}(df_train[:, input_cols])
train_targets = df_train[:, target_col] .== 1 

# ============================================================================
# 4. TRAINING POTENZIATO
# ============================================================================
println("\n--- FASE 2: Training con attivazione ReLU (Più aggressiva)... ---")

# MODIFICA FONDAMENTALE:
# Usiamo [relu, relu] invece di default. Aiuta a non bloccarsi su "Medio".
topology = [64, 32]  # Aumentato ancora i neuroni
learning_rate = 0.005 # Leggermente più basso per stabilità
max_epochs = 500     
k_folds = 5      

# Definiamo le funzioni di attivazione per i layer nascosti (ReLU è standard moderno)
functions = [Flux.relu, Flux.relu]

cv_indices = crossvalidation(train_targets, k_folds)

results = ANNCrossValidation(
    topology, 
    (train_inputs, train_targets), 
    cv_indices;
    maxEpochs=max_epochs, 
    learningRate=learning_rate,
    numExecutions=1,
    transferFunctions=functions # Passiamo ReLU qui
)

(meanAcc, stdAcc), (meanErr, stdErr), (meanSens, stdSens), 
(meanSpec, stdSpec), (meanPPV, stdPPV), _, (meanF1, stdF1), confMatrix = results

# ============================================================================
# 5. REPORT PRESTAZIONI
# ============================================================================
println("\n=== REPORT PRESTAZIONI ===")
println("1. Accuratezza: $(round(meanAcc * 100, digits=2))%")
println("2. Sensibilità: $(round(meanSens * 100, digits=2))%")
println("3. Specificità: $(round(meanSpec * 100, digits=2))%")
println("4. F1 Score: $(round(meanF1, digits=2))")

println("\nMatrice di Confusione:")
display(confMatrix)

# ============================================================================
# 6. GENERAZIONE FILE OUTPUT
# ============================================================================
println("\n--- FASE 3: Analisi Finale e Export ---")

normParams = calculateMinMaxNormalizationParameters(train_inputs)
train_inputs_norm = normalizeMinMax(train_inputs, normParams)
train_targets_matrix = reshape(train_targets, :, 1)

println("Rieducazione modello finale...")
final_model, _ = trainClassANN(topology, (train_inputs_norm, train_targets_matrix); 
                               maxEpochs=max_epochs, 
                               learningRate=learning_rate,
                               transferFunctions=functions) # Anche qui ReLU

# Preparazione dati originali
all_inputs = Matrix{Float32}(df_original[:, input_cols])
all_inputs_norm = normalizeMinMax(all_inputs, normParams)

println("Calcolo probabilità...")
probabilities = final_model(all_inputs_norm')' 

# DEBUG: Vediamo se sono ancora tutti uguali
println("Statistiche Probabilità calcolate:")
println("  Min: $(minimum(probabilities))")
println("  Max: $(maximum(probabilities))")
println("  Media: $(mean(probabilities))")

# LOGICA RISCHIO
function get_risk_label(p)
    if p < 0.20 return "Basso"
    elseif p < 0.80 return "Medio/Sospetto"
    else return "ALTO / CRITICO"
    end
end

risk_levels = get_risk_label.(probabilities)

output_df = DataFrame(
    Transaction_ID = df_original[:, id_col],
    Fraud_Probability = round.(vec(probabilities), digits=4), 
    Risk_Level = vec(risk_levels)
)

output_filename = "report_transazioni_rischio.csv"
CSV.write(output_filename, output_df)

println("✅ Fatto! File salvato.")

--- FASE 1: Caricamento e Pulizia Dati ---
Feature utilizzate: ["LogAmount", "Transaction Hour", "IP_Frequency", "Account Age Days"]
Dataset Bilanciato: 150120 righe totali.

--- FASE 2: Training con attivazione ReLU (Più aggressiva)... ---

Fold 1/5
  Execution 1/1

Fold 2/5
  Execution 1/1

Fold 3/5
  Execution 1/1

Fold 4/5
  Execution 1/1

Fold 5/5
  Execution 1/1

=== REPORT PRESTAZIONI ===
1. Accuratezza: 74.1%
2. Sensibilità: 81.2%
3. Specificità: 67.0%
4. F1 Score: 0.76

Matrice di Confusione:


2×2 Matrix{Float64}:
 50291.0  24769.0
 14108.0  60952.0


--- FASE 3: Analisi Finale e Export ---
Rieducazione modello finale...
Calcolo probabilità...
Statistiche Probabilità calcolate:
  Min: 0.032721326
  Max: 0.99994195
  Media: 0.35583836
✅ Fatto! File salvato.
